# Diverse-model stacking

This notebook tests whether adding ExtraTrees or elastic-net logistic regression improves the chronological stack of HistGradientBoosting, CatBoost and LightGBM. The new models use different learning assumptions: ExtraTrees averages highly randomised trees, while elastic-net logistic regression fits a regularised linear relationship.

ExtraTrees uses 400 trees with a minimum leaf size of 3. Elastic-net logistic regression uses `C=0.1` and `l1_ratio=0.5`. Each model is added separately to the existing stack, without a hyperparameter search. The stacking coefficients are fitted on out-of-fold predictions from steps 10–14, 15–19 and 20–24. We then evaluate steps 25–29 and 30–35, as well as two longer prediction horizons.

The validation windows have also informed earlier experiments, so these comparisons are useful for model selection but are not an independent estimate of future performance.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'/'raw'
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
features=[c for c in train if c.startswith('feat_')]
labelled=train.dropna(subset=['label']).copy()
assert labelled.label.isin([0,1]).all()
assert 'time_step' in train and all(f in test for f in features)
assert len(features)>0
print('Labelled',len(labelled),'test',len(test),'features',len(features))

Labelled 31235 test 15329 features 165


In [2]:
SEEDS=[11,42,73,101,202]
HGB_PARAMS=dict(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=42)
CAT_PARAMS=dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,loss_function='Logloss',verbose=False,thread_count=-1)
LGB_PARAMS=dict(n_estimators=700,learning_rate=0.045,num_leaves=31,max_depth=-1,reg_lambda=5.0,random_state=42,verbosity=-1,n_jobs=-1)
META_C=0.1
CANDIDATES=['extra_trees','elastic_net']
META_WINDOWS=[(10,14),(15,19),(20,24)]
EVAL_WINDOWS=[(25,29),(30,35)]

# Each fold fits preprocessing only on its past training observations.
def fit_predict(tr,va):
    X=tr[features]; y=tr.label.to_numpy(dtype=int); V=va[features]
    if len(np.unique(y))!=2: raise ValueError('Training fold has only one class')
    h=HistGradientBoostingClassifier(**HGB_PARAMS).fit(X,y)
    ph=h.predict_proba(V)[:,1]
    pcs=[]
    for seed in SEEDS:
        m=CatBoostClassifier(**CAT_PARAMS,random_seed=seed).fit(X,y)
        pcs.append(m.predict_proba(V)[:,1])
    pc=np.mean(pcs,axis=0)
    l=LGBMClassifier(**LGB_PARAMS).fit(X,y)
    pl=l.predict_proba(V)[:,1]
    et=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),
                     ExtraTreesClassifier(n_estimators=400,min_samples_leaf=3,
                                          max_features='sqrt',n_jobs=-1,random_state=42))
    et.fit(X,y); pe=et.predict_proba(V)[:,1]
    en=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),
                     StandardScaler(),
                     LogisticRegression(penalty='elasticnet',solver='saga',l1_ratio=0.5,
                                        C=0.1,max_iter=3000,tol=1e-3,random_state=42))
    en.fit(X,y); pn=en.predict_proba(V)[:,1]
    return np.column_stack([ph,pc,pl,pe,pn])

# Avoid accidental reuse of stale prediction caches; this experiment computes fresh predictions.
from functools import lru_cache
@lru_cache(maxsize=None)
def get_window(origin,lo,hi):
    assert origin < lo <= hi
    tr=labelled[labelled.time_step<=origin]
    va=labelled[labelled.time_step.between(lo,hi)]
    if tr.empty or va.empty: raise ValueError(f'Empty split {origin}, {lo}-{hi}')
    print(f'Fitting origin={origin}, validation={lo}-{hi}, n={len(va)}',flush=True)
    return va.label.to_numpy(dtype=int), fit_predict(tr,va)

def ap(y,p): return average_precision_score(y,p)
def roc(y,p): return roc_auc_score(y,p)
def fit_meta(P,y,cols):
    return LogisticRegression(C=META_C,max_iter=2000,solver='lbfgs').fit(P[:,cols],y)

def predict_models(P,meta,cols): return meta.predict_proba(P[:,cols])[:,1]

## Chronological stacking comparison

The three-model baseline uses HGB, CatBoost and LightGBM predictions. Two alternative stacks add either ExtraTrees or elastic-net logistic regression. All three meta-models are trained on the same chronological out-of-fold observations, allowing the effect of each additional model to be compared directly.

The baseline is refitted in this notebook rather than read from an earlier submission file.

In [3]:
meta_blocks=[]; meta_labels=[]
for lo,hi in META_WINDOWS:
    y,P=get_window(lo-1,lo,hi)
    meta_blocks.append(P); meta_labels.append(y)
Pmeta=np.vstack(meta_blocks); ymeta=np.concatenate(meta_labels)
COLS={'exp18':[0,1,2],'plus_extra_trees':[0,1,2,3],'plus_elastic_net':[0,1,2,4]}
metas={name:fit_meta(Pmeta,ymeta,cols) for name,cols in COLS.items()}
print('Chronological meta rows:',len(ymeta))
print('Meta coefficients:',{k:v.coef_.round(3).tolist() for k,v in metas.items()})

Fitting origin=9, validation=10-14, n=3400


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fitting origin=14, validation=15-19, n=3093


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fitting origin=19, validation=20-24, n=5617


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Chronological meta rows: 12110
Meta coefficients: {'exp18': [[2.414, 3.413, 2.713]], 'plus_extra_trees': [[2.132, 3.058, 2.477, 1.523]], 'plus_elastic_net': [[2.332, 3.295, 2.643, 0.516]]}


In [4]:
short_rows=[]; short_predictions={}
for lo,hi in EVAL_WINDOWS:
    y,P=get_window(lo-1,lo,hi)
    short_predictions[(lo,hi)]=(y,P)
    baseline=predict_models(P,metas['exp18'],COLS['exp18'])
    for name,cols in COLS.items():
        p=predict_models(P,metas[name],cols)
        short_rows.append(dict(period=f'{lo}-{hi}',model=name,pr_auc=ap(y,p),roc_auc=roc(y,p),
                               delta_pr=ap(y,p)-ap(y,baseline)))
short=pd.DataFrame(short_rows)
display(short.round(6))

Fitting origin=24, validation=25-29, n=2775


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fitting origin=29, validation=30-35, n=4854


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,period,model,pr_auc,roc_auc,delta_pr
0,25-29,exp18,0.993799,0.997780,0.000000
1,25-29,plus_extra_trees,0.994181,0.997828,0.000382
2,25-29,plus_elastic_net,0.992059,0.996893,-0.001740
3,30-35,exp18,0.992635,0.998414,0.000000
4,30-35,plus_extra_trees,0.991969,0.998322,-0.000665
5,30-35,plus_elastic_net,0.991116,0.997869,-0.001519


## Longer prediction horizons

To examine temporal generalisation, we train using information available through step 14 and evaluate steps 15–28, then train through step 21 and evaluate steps 22–35. For each simulation, the stacking model uses only earlier out-of-fold windows. We also compare performance on the later half of each evaluation period, where distribution shift may be more pronounced.

ExtraTrees improves full-period PR-AUC from 0.922285 to 0.923465 in the first simulation, but reduces it from 0.800021 to 0.786333 in the second. The elastic-net addition performs worse than the baseline in both simulations.

In [5]:
LONG_FOLDS=[(14,15,28),(21,22,35)]
long_rows=[]
for origin,lo,hi in LONG_FOLDS:
    windows=[(a,b) for a,b in META_WINDOWS if b<=origin]
    if not windows: raise ValueError('No prior OOF windows')
    parts=[]; labs=[]
    for a,b in windows:
        yy,PP=get_window(a-1,a,b)
        parts.append(PP);labs.append(yy)
    Pm=np.vstack(parts); ym=np.concatenate(labs)
    models={name:fit_meta(Pm,ym,cols) for name,cols in COLS.items()}
    y,P=get_window(origin,lo,hi)
    base=predict_models(P,models['exp18'],COLS['exp18'])
    # Evaluate late seven steps using row-level time_step mask, not assuming equal counts per step.
    va=labelled[labelled.time_step.between(lo,hi)]
    late=(va.time_step.to_numpy()>=lo+7)
    for name,cols in COLS.items():
        p=predict_models(P,models[name],cols)
        long_rows.append(dict(origin=origin,period=f'{lo}-{hi}',model=name,
                              full_pr=ap(y,p),delta_full=ap(y,p)-ap(y,base),
                              late_pr=ap(y[late],p[late]),
                              delta_late=ap(y[late],p[late])-ap(y[late],base[late]),
                              full_roc=roc(y,p)))
long=pd.DataFrame(long_rows)
display(long.round(6))

Fitting origin=14, validation=15-28, n=10311


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fitting origin=21, validation=22-35, n=11705


.../site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,origin,period,model,full_pr,delta_full,late_pr,delta_late,full_roc
0,14,15-28,exp18,0.922285,0.000000,0.837417,0.000000,0.956465
1,14,15-28,plus_extra_trees,0.923465,0.001180,0.842171,0.004754,0.961082
2,14,15-28,plus_elastic_net,0.912964,-0.009321,0.827142,-0.010275,0.961916
3,21,22-35,exp18,0.800021,0.000000,0.753918,0.000000,0.874063
4,21,22-35,plus_extra_trees,0.786333,-0.013689,0.740202,-0.013715,0.819022
5,21,22-35,plus_elastic_net,0.793270,-0.006751,0.749390,-0.004527,0.872415


## Prediction diversity

Spearman correlation measures how similarly each additional model ranks transactions compared with the existing stack. ExtraTrees has correlations of 0.8488 and 0.7822 across the two recent windows; elastic net has lower correlations of 0.7425 and 0.6861. Lower correlation alone is not enough to justify inclusion, since a model must also contribute useful predictions.

We retain a candidate only if it improves both long-horizon PR-AUC scores, avoids losses in their later halves, and does not materially worsen either recent window. Neither addition meets these conditions. The error-overlap diagnostic is undefined for some comparisons because no transactions fall into one or both sets being compared.

In [6]:
from scipy.stats import spearmanr
for key,(y,P) in short_predictions.items():
    base=predict_models(P,metas['exp18'],COLS['exp18'])
    print('Period',key)
    for idx,name in [(3,'ExtraTrees'),(4,'Elastic Net')]:
        corr=spearmanr(base,P[:,idx]).statistic
        # Bottom 50% of fraud predictions is a descriptive low-ranked-fraud set.
        fraud=(y==1)
        bad_base=fraud & (base<np.median(base))
        bad_new=fraud & (P[:,idx]<np.median(P[:,idx]))
        union=np.sum(bad_base|bad_new)
        overlap=np.sum(bad_base&bad_new)/union if union else np.nan
        print(name,'standalone AP',round(ap(y,P[:,idx]),6),
              'Spearman vs baseline',round(corr,4),'low-ranked-fraud Jaccard',round(overlap,4))

checks=[]
for name in ['plus_extra_trees','plus_elastic_net']:
    lr=long[long.model==name]
    sr=short[short.model==name]
    eligible=bool((lr.delta_full>0).all() and (lr.delta_late>=0).all() and (sr.delta_pr>=-0.0005).all())
    checks.append(dict(model=name,mean_long_gain=lr.delta_full.mean(),worst_long_gain=lr.delta_full.min(),
                       worst_late_gain=lr.delta_late.min(),worst_short_gain=sr.delta_pr.min(),eligible=eligible))
summary=pd.DataFrame(checks).sort_values('mean_long_gain',ascending=False)
display(summary.round(6))
print('Recommended next step:', 'Review eligible candidate(s) before any Kaggle submission' if summary.eligible.any() else 'Retain Experiment 18 (public 0.94914).')

Period (25, 29)
ExtraTrees standalone AP 0.986883 Spearman vs baseline 0.8488 low-ranked-fraud Jaccard nan
Elastic Net standalone AP 0.661339 Spearman vs baseline 0.7425 low-ranked-fraud Jaccard 0.0
Period (30, 35)
ExtraTrees standalone AP 0.982068 Spearman vs baseline 0.7822 low-ranked-fraud Jaccard nan
Elastic Net standalone AP 0.594632 Spearman vs baseline 0.6861 low-ranked-fraud Jaccard 0.0


,model,mean_long_gain,worst_long_gain,worst_late_gain,worst_short_gain,eligible
0,plus_extra_trees,-0.006254,-0.013689,-0.013715,-0.000665,False
1,plus_elastic_net,-0.008036,-0.009321,-0.010275,-0.001740,False


Recommended next step: Retain Experiment 18 (public 0.94914).


## Submission generation

The submission code is disabled because neither candidate passed the validation criteria. If a candidate had qualified, the code would refit the base models on labelled transactions through step 35 and predict the unlabelled test transactions. The existing stacking submission remains unchanged.

In [7]:
MAKE_SUBMISSION=False
CHOSEN_MODEL=None  # Set to 'plus_extra_trees' or 'plus_elastic_net' ONLY if eligible.
if MAKE_SUBMISSION:
    eligible_models=set(summary.loc[summary.eligible,'model'])
    if CHOSEN_MODEL not in eligible_models:
        raise ValueError(f'Choose an eligible model from {sorted(eligible_models)}')
    assert test.time_step.min()>labelled.time_step.max()
    assert list(sample.columns)==['index','target']
    Ptest=fit_predict(labelled,test)
    pred=predict_models(Ptest,metas[CHOSEN_MODEL],COLS[CHOSEN_MODEL])
    assert len(pred)==len(test)==len(sample) and np.isfinite(pred).all()
    sub=sample.copy(); sub['target']=pred
    out=ROOT/'submissions'/f'experiment26_{CHOSEN_MODEL}.csv'
    out.parent.mkdir(exist_ok=True,parents=True)
    if out.exists(): raise FileExistsError(out)
    sub.to_csv(out,index=False)
    print('Saved',out)
else:
    print('Submission disabled. Keep Experiment 18 as current best.')

Submission disabled. Keep Experiment 18 as current best.


## Findings

The additional models do not provide a consistent improvement over the three-model stack. ExtraTrees slightly improves PR-AUC on steps 25–29 (0.993799 to 0.994181), but falls behind on steps 30–35 (0.992635 to 0.991969) and on the second long-horizon simulation. Elastic-net logistic regression reduces PR-AUC in both recent windows and both long-horizon simulations.

The results suggest that a different model family is not necessarily a useful addition to an ensemble. Although ExtraTrees and elastic net produce different rankings, neither improves temporal robustness sufficiently to replace the existing stacking model. No Kaggle submission was generated from this notebook.